# Step 2: Explore the Green Taxi data

The goal is to find out what needs cleaning before revenue per day can be added up.
This notebook is only for experimenting. The real cleaning happens in the Step 3 pipeline.

1. Load one file
2. Find the columns needed: when the trip happened and what was paid
3. Check them: data types, missing values, value ranges for dates and amounts
4. Note what needs cleaning

In [1]:
from pathlib import Path

import pandas as pd

RAW_DIR = Path.cwd().parent / "data" / "raw" if Path.cwd().name == "notebooks" else Path("data/raw")
sorted(p.name for p in RAW_DIR.glob("*.parquet"))

['green_tripdata_2025-01.parquet',
 'green_tripdata_2025-02.parquet',
 'green_tripdata_2025-03.parquet']

## 2.1 Load one file

Start with January 2025 to see what one month looks like.

In [2]:
jan = pd.read_parquet(RAW_DIR / "green_tripdata_2025-01.parquet")
print(f"{len(jan):,} rows x {jan.shape[1]} columns")
jan.head()

48,326 rows x 21 columns


,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,...,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee
0,2,2025-01-01 00:03:01,2025-01-01 00:17:12,N,1.0,75,235,1.0,5.93,24.70,...,0.5,6.80,0.00,NaN,1.0,34.00,1.0,1.0,0.00,0.0
1,2,2025-01-01 00:19:59,2025-01-01 00:25:52,N,1.0,166,75,1.0,1.32,8.60,...,0.5,0.00,0.00,NaN,1.0,11.10,2.0,1.0,0.00,0.0
2,2,2025-01-01 00:05:29,2025-01-01 00:07:21,N,5.0,171,73,1.0,0.41,25.55,...,0.0,0.00,0.00,NaN,1.0,26.55,2.0,2.0,0.00,0.0
3,2,2025-01-01 00:52:24,2025-01-01 01:07:52,N,1.0,74,223,1.0,4.12,21.20,...,0.5,6.13,6.94,NaN,1.0,36.77,1.0,1.0,0.00,0.0
4,2,2025-01-01 00:25:05,2025-01-01 01:01:10,N,1.0,66,158,1.0,4.71,33.80,...,0.5,7.81,0.00,NaN,1.0,46.86,1.0,1.0,2.75,0.0


## 2.2 Find the columns needed

| Question | Column | Why |
|---|---|---|
| When did the trip happen? | `lpep_pickup_datetime` | Count each trip on the day it **started** (`lpep` = Green Taxi, "Licensed Passenger Enhancement Program") |
| What was paid? | `total_amount` | Everything the passenger was charged: fare + tips + tolls + taxes + surcharges |
| (alternative) | `fare_amount` | Only the metered fare, without tips, tolls and surcharges |
| (context) | `payment_type` | Helps explain odd amounts: 1 = Credit card, 2 = Cash, 3 = No charge, 4 = Dispute, 5 = Unknown, 6 = Voided trip |

**Choice:** revenue = `total_amount`, grouped by the date of `lpep_pickup_datetime`.

In [3]:
cols = ["lpep_pickup_datetime", "total_amount", "fare_amount", "payment_type"]
jan[cols].head()

,lpep_pickup_datetime,total_amount,fare_amount,payment_type
0,2025-01-01 00:03:01,34.00,24.70,1.0
1,2025-01-01 00:19:59,11.10,8.60,2.0
2,2025-01-01 00:05:29,26.55,25.55,2.0
3,2025-01-01 00:52:24,36.77,21.20,1.0
4,2025-01-01 00:25:05,46.86,33.80,1.0


## 2.3 Check the columns

### Data types

In [4]:
jan[cols].dtypes

lpep_pickup_datetime    datetime64[us]
total_amount                   float64
fare_amount                    float64
payment_type                   float64
dtype: object

Both columns already have the right types: the pickup time is a `datetime64` and the amounts are `float64`. No converting needed.

### Missing values

In [5]:
jan.isna().sum().loc[lambda s: s > 0]

store_and_fwd_flag       1836
RatecodeID               1836
passenger_count          1836
ehail_fee               48326
payment_type             1836
trip_type                1843
congestion_surcharge     1836
cbd_congestion_fee       1836
dtype: int64

`lpep_pickup_datetime` and `total_amount` have **no missing values**. About 1,836 rows are missing other columns,
such as `payment_type` and `passenger_count`. That doesn't matter for revenue, so those rows are kept.
`ehail_fee` is always empty, so it can be ignored.

### Value range: dates

In [6]:
print("earliest pickup:", jan["lpep_pickup_datetime"].min())
print("latest pickup:  ", jan["lpep_pickup_datetime"].max())

earliest pickup: 2024-12-25 23:13:15
latest pickup:   2025-02-05 18:46:24


The January file has pickups **before January 2025**. Check all three files to see how many fall outside the Q1 2025 period:

In [7]:
all_trips = pd.concat(
    [pd.read_parquet(p) for p in sorted(RAW_DIR.glob("green_tripdata_2025-*.parquet"))],
    ignore_index=True,
)
pickup = all_trips["lpep_pickup_datetime"]
outside_q1 = all_trips[(pickup < "2025-01-01") | (pickup >= "2025-04-01")]

print(f"all trips:      {len(all_trips):,}")
print(f"outside Q1 2025: {len(outside_q1):,}")
outside_q1[["lpep_pickup_datetime", "total_amount"]].sort_values("lpep_pickup_datetime")

all trips:      146,486
outside Q1 2025: 11


,lpep_pickup_datetime,total_amount
320,2024-12-25 23:13:15,43.20
4108,2024-12-29 01:23:41,72.94
251,2024-12-31 19:56:14,13.32
9,2024-12-31 22:42:13,25.00
10,2024-12-31 23:01:11,13.81
73,2024-12-31 23:30:28,11.90
141386,2025-04-01 01:05:44,18.11
141851,2025-04-01 11:46:44,16.70
142867,2025-04-01 21:01:47,16.25
142913,2025-04-01 22:27:14,12.50


Only a few trips are outside Jan 1 to Mar 31, 2025, but each one would add a stray day (for example 2024-12-25)
to the result. Trips listed in one month's file but picked up in the next month (for example Jan 31 to Feb 1)
are fine: grouping by the pickup date puts them on the right day.

### Value range: amounts

In [8]:
all_trips["total_amount"].describe()

count    146486.00000
mean         23.18034
std          16.48806
min        -473.10000
25%          13.80000
50%          18.90000
75%          27.18000
max         642.14000
Name: total_amount, dtype: float64

In [9]:
amount = all_trips["total_amount"]
print(f"negative total_amount: {(amount < 0).sum():,}")
print(f"zero total_amount:     {(amount == 0).sum():,}")
print(f"over $500:             {(amount > 500).sum():,}")

negative total_amount: 478
zero total_amount:     254
over $500:             2


Which payment types do the negative amounts come from?

In [10]:
payment_names = {1: "Credit card", 2: "Cash", 3: "No charge", 4: "Dispute", 5: "Unknown", 6: "Voided"}
(
    all_trips.loc[amount < 0, "payment_type"]
    .map(payment_names)
    .value_counts(dropna=False)
)

payment_type
No charge    337
Dispute      127
Cash          14
Name: count, dtype: int64

Most negative amounts are **No charge** or **Dispute** trips. They are reversals or voided trips, not real revenue.
The largest amounts (up to about $640) are high but possible for long trips, so they are kept.

## 2.4 What needs cleaning

| # | Problem | Rule for Step 3 |
|---|---|---|
| 1 | Pickups outside Jan 1 to Mar 31, 2025 | **Drop**: they belong to other periods and would add stray days |
| 2 | `total_amount` below 0 (mostly "No charge" / "Dispute") | **Drop**: reversals and voided trips, not revenue |
| 3 | `total_amount` exactly 0 | **Drop**: no revenue, and they would pull down the average amount per trip |
| 4 | Missing `payment_type`, `passenger_count`, and similar | **Keep**: not needed for revenue |
| 5 | Large amounts (up to about $640) | **Keep**: plausible long trips |

Step 3 should print the row count before and after each rule, so it's clear how much each rule removed.
Here's a preview:

In [11]:
in_q1 = (pickup >= "2025-01-01") & (pickup < "2025-04-01")
positive = amount > 0
clean = all_trips[in_q1 & positive]

print(f"raw rows:              {len(all_trips):,}")
print(f"  outside Q1 removed:  {(~in_q1).sum():,}")
print(f"  amount <= 0 removed: {(in_q1 & ~positive).sum():,}")
print(f"clean rows:            {len(clean):,}")

raw rows:              146,486
  outside Q1 removed:  11
  amount <= 0 removed: 732
clean rows:            145,743


Quick check of the end result: revenue per day using the clean rows (Step 3 will do this properly).

In [12]:
daily = (
    clean.groupby(clean["lpep_pickup_datetime"].dt.date)["total_amount"]
    .sum()
    .round(2)
    .rename("revenue")
)
print(f"{len(daily)} days, from {daily.index.min()} to {daily.index.max()}")
daily.head()

90 days, from 2025-01-01 to 2025-03-31


lpep_pickup_datetime
2025-01-01    25195.01
2025-01-02    34278.18
2025-01-03    36533.99
2025-01-04    27977.60
2025-01-05    25838.19
Name: revenue, dtype: float64